# FAISS Project – SIFT Starter Experiment

This notebook loads the SIFT dataset, builds exact ground truth, and runs a first IVFPQ experiment with Recall@10 and QPS.

In [1]:
import sys
from pathlib import Path

import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().resolve().parent
else:
    PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

C:\Users\vaspn\Downloads\Vector-Search-FAISS


In [3]:
from src.data_loading import load_sift
from src.benchmarks import build_ground_truth, benchmark_index
from src.metrics import recall_at_k

## Load SIFT

In [4]:
xb, xq, xlearn, gt = load_sift()

print("xb:", xb.shape, xb.dtype)
print("xq:", xq.shape, xq.dtype)
print("xlearn:", xlearn.shape, xlearn.dtype)
print("gt:", gt.shape, gt.dtype)

xb: (1000000, 128) float32
xq: (10000, 128) float32
xlearn: (100000, 128) float32
gt: (10000, 100) int32


## Exact Ground Truth

In [5]:
k = 10
d = xb.shape[1]
metric = faiss.METRIC_L2

D_gt, I_gt = build_ground_truth(xb, xq, k, metric)
print("D_gt shape:", D_gt.shape)
print("I_gt shape:", I_gt.shape)

D_gt shape: (10000, 10)
I_gt shape: (10000, 10)


## First IVFPQ Experiment

In [6]:
factory = "IVF4096,PQ32x8"
index = faiss.index_factory(d, factory, metric)

train_data = xlearn if xlearn is not None and len(xlearn) > 0 else xb
if not index.is_trained:
    index.train(train_data)

index.add(xb)

In [7]:
ps = faiss.ParameterSpace()
ps.set_index_parameter(index, "nprobe", 16)

D, I, total_time, qps = benchmark_index(index, xq, k)
recall = recall_at_k(I, I_gt, k)

print("Factory:", factory)
print("Recall@10:", recall)
print("QPS:", qps)
print("Total search time:", total_time)

Factory: IVF4096,PQ32x8
Recall@10: 0.66783
QPS: 12973.894189419761
Total search time: 0.7707786000100896


## Notes

This is the first working benchmark. Next steps are parameter sweeps for IVFPQ, HNSWSQ, and LSH.